# 7. A theory of the floor

Notebooks 04–05 split a life's growth variance into a fitness floor $V_F$ and a demographic part $B$, and showed that the
ratio curve bends only if the floor differs between firms of equal size. This notebook explains the floor. The thesis did the
same for the deterministic limit (immigration floor $\lambda$, listing threshold); here the model is the demographic one of
the paper, with exits at $S=0$ and newborns at $S=D$. Reference point and the two stored economies per graph as in 04–05.

1. **What the floor is made of.** $V_F=V_{\rm core}+(E_{\rm in}+E_{\rm out})/n$: fluctuations on the plateau, plus the climb from the
   newborn size and the final decline, spread over the life's $n$ growth rates. At fixed size the floor varies mainly
   because the life length does; the degree sets how large the floor is against the noise.
2. **How long a firm lives.** Established firms die in dips of their field, at Rice's downcrossing rate thinned by the
   Feller extinction probability $P_{\rm kill}=e^{-2/(DJ)}$, $J=\int dt/S$ along the dip. No fitted parameter.
3. **What the field must provide.** A Gaussian field with each node's mean $m_i$ and amplitude $f_i$ and one correlation
   function reproduces the life panel. Both spread at equal degree, and both spreads carry the bend; the shape of the spread
   of the means (close to Gaussian) does not matter. The four-constant cavity law keeps most of the bend for long lives.
4. **Closing the chain.** The two-time heterogeneous mean-field theory, solved by iterating single sites, gives the static
   order parameters, the turnover and the long-life bend with no input from the network simulation; the dynamic variance
   is not yet converged.

In [1]:
import numpy as np
from scipy import stats
import economy as eco
from economy import lives, zeta_bins, DATA, MSB_RATIO

FLOOR = DATA / "demographic" / "floor"
FLOOR.mkdir(parents=True, exist_ok=True)
QS = np.arange(1, 5)
f2 = lambda a: " ".join(f"{x:5.2f}" for x in a)


def life_segments(L, nmin):
    T = L.shape[1]
    return [(i, seg) for i in range(L.shape[0]) for seg in np.split(np.arange(T), np.flatnonzero(np.diff(L[i])) + 1)
            if seg.size >= nmin + 1]


def floor_parts(kind="powerlaw", nmin=20):
    """Per life with >= nmin growth rates: the noise-free path along the measured field from the life's first size, its
    growth variance V_F, the core (between the first and last time the noise-free size is at least half its median),
    the sums of squared noise-free growth over the entry climb and the exit decline, and the inputs of the logistic
    transient formulas."""
    path = FLOOR / f"parts_{kind}.npz"
    if not path.exists():
        rows = []
        for seed in eco.SEEDS:
            S, L, deg, _, Phi = eco.reference_economy(seed, kind)
            T = S.shape[1]
            lv = life_segments(L, nmin)
            lv.sort(key=lambda x: x[1].size)
            for c0 in range(0, len(lv), 20000):
                part = lv[c0:c0 + 20000]
                ph, x0 = np.zeros((len(part), part[-1][1].size)), np.empty(len(part))
                for k, (i, seg) in enumerate(part):
                    ph[k, :seg.size], ph[k, seg.size:], x0[k] = Phi[i, seg], Phi[i, seg[-1]], S[i, seg[0]]
                det = eco.noise_free(ph, np.log(x0))
                for k, (i, seg) in enumerate(part):
                    m, x, f = seg.size, S[i, seg], Phi[i, seg]
                    yd = det[k, :m]; sd = np.exp(yd); gd = np.diff(yd)
                    up = np.flatnonzero(sd >= 0.5 * np.median(sd)); a, b = up[0], up[-1]
                    core = gd[a:b] if b - a >= 10 else gd
                    h = core.size // 2
                    rows.append((x.mean(), np.diff(np.log(x)).var(), gd.var(), core.var(),
                                 np.sum(gd[:a] ** 2), np.sum(gd[b:] ** 2),
                                 np.mean(eco.D / x[:-1] ** 2 * (1 - np.exp(-x[:-1] * eco.DT))), deg[i], m - 1,
                                 core[:h].var(), core[h:].var(), sd[a:b + 1].mean(), x0[k], sd[-1], -f[b:].mean(),
                                 float(seg[0] > 0 and seg[-1] < T - 1)))
        cols = "Sbar varg VF Vcore SSent SSext B deg n Vh1 Vh2 h x0 Send phi complete".split()
        np.savez(path, rows=np.array(rows), cols=cols)
    z = np.load(path)
    return {k: z["rows"][:, j] for j, k in enumerate(z["cols"])}

## What the floor is made of

A life with at least 20 growth rates usually starts as a newborn at $S\simeq D$ and ends at an exit. Its noise-free path
climbs to a plateau, follows the field there, and collapses in the final dip. Split the noise-free growth rates at the first
and last time the noise-free size is at least half its median: the **core** is the plateau, the rest is the **entry climb** and
the **exit decline**. Within a size bin, replace one part by its bin mean at a time and recompute the exponents with the
demographic part $B$ at its bin mean, as in notebook 05.

In [2]:
def bins(Sb, n=25):
    P = np.array_split(np.argsort(Sb), n)
    return P, np.log([Sb[p].mean() for p in P])


def binmean(x, P):
    y = np.empty_like(x)
    for p in P:
        y[p] = x[p].mean()
    return y


def zeta_of(Y, P, lx):
    return np.array([-np.polyfit(lx, np.log([np.mean(Y[p] ** (q / 2)) for p in P]), 1)[0] for q in QS])


for kind in ("powerlaw", "regular"):
    d = floor_parts(kind)
    Sb, VF, Vc, B, n, kap = d["Sbar"], d["VF"], d["Vcore"], d["B"], d["n"], d["deg"] / d["deg"].mean()
    Vt = VF - Vc
    P, lx = bins(Sb)
    Bs, Vcs, Vts = binmean(B, P), binmean(Vc, P), binmean(Vt, P)
    print(f"=== {kind} graph: {Sb.size} lives with >= 20 growth rates")
    for name, Y in (("measured", d["varg"]), ("own floor", VF + Bs), ("own core only", Vc + Vts + Bs),
                    ("own entry/exit only", Vcs + np.maximum(Vt, 0) + Bs), ("all at bin means", binmean(VF, P) + Bs)):
        z = zeta_of(Y, P, lx)
        print(f"  {name:20s} zeta_1 {z[0]:.3f}   ratios {f2(z[1:] / z[0])}")
    print("  by fifth (within-bin values averaged):  Sbar   n    V_F   core  entry/exit share   1-w   "
          "Var ln V_F   Var ln n   R2(ln n)  R2(ln kappa)")
    for i in range(0, 25, 5):
        r = []
        for p in P[i:i + 5]:
            Y = VF[p] + B[p].mean()
            r.append((Sb[p].mean(), n[p].mean(), VF[p].mean(), Vc[p].mean(), Vt[p].mean() / VF[p].mean(),
                      np.mean(VF[p] / Y), np.log(VF[p]).var(), np.log(n[p]).var(),
                      np.corrcoef(np.log(VF[p]), np.log(n[p]))[0, 1] ** 2,
                      np.corrcoef(np.log(VF[p]), np.log(kap[p]))[0, 1] ** 2 if kap.std() > 0 else 0.0))
        r = np.mean(r, 0)
        print(f"  {'':40s}{r[0]:5.2f} {r[1]:5.0f}  {r[2]:.3f}  {r[3]:.3f}      {r[4]:.2f}           {r[5]:.2f}     "
              f"{r[6]:.2f}        {r[7]:.2f}      {r[8]:.2f}       {r[9]:.2f}")

=== powerlaw graph: 198840 lives with >= 20 growth rates
  measured             zeta_1 0.190   ratios  1.75  2.30  2.70
  own floor            zeta_1 0.230   ratios  1.85  2.55  3.07
  own core only        zeta_1 0.215   ratios  1.98  2.94  3.85
  own entry/exit only  zeta_1 0.225   ratios  1.90  2.69  3.33
  all at bin means     zeta_1 0.213   ratios  2.00  3.00  4.00
  by fifth (within-bin values averaged):  Sbar   n    V_F   core  entry/exit share   1-w   Var ln V_F   Var ln n   R2(ln n)  R2(ln kappa)
                                           0.36    29  0.113  0.084      0.25           0.20     0.50        0.08      0.08       0.14
                                           0.71    39  0.133  0.086      0.36           0.31     0.47        0.18      0.26       0.19
                                           1.07    53  0.137  0.076      0.45           0.37     0.53        0.31      0.38       0.24
                                           1.55    79  0.134  0.063      0.53        

In [3]:
d = floor_parts("powerlaw")
P, _ = bins(d["Sbar"])
r = [np.corrcoef(np.log(d["n"][p]), np.log(d["deg"][p]))[0, 1] for p in P]
print("correlation of ln n with ln kappa within size bins, by fifth (power-law graph):",
      f2([np.mean(r[i:i + 5]) for i in range(0, 25, 5)]))

correlation of ln n with ln kappa within size bins, by fifth (power-law graph): -0.16 -0.25 -0.32 -0.38 -0.49


Reading the output:

- **The floor is two opposite trends.** The core falls with size, while the entry and exit share grows, from about a quarter
  of $V_F$ in the smallest fifth to about 60% in the largest: large firms climb further from $D$ and collapse from higher. The
  flat floor of notebook 04 is their sum.
- **The bend comes from the entry and exit part.** With only the core heterogeneous the ratios are close to $q$; with only the
  entry and exit heterogeneous most of the bend is back.
- **Life length is the main source of floor dispersion at fixed size**: it explains more than the degree in every fifth but
  the smallest on the power-law graph, and up to half of it on the regular graph, where the degree explains nothing.
  $\operatorname{Var}(\ln n\mid\bar S)$ grows with size, which is why the bend is strongest among large firms.
- **What the degree does.** On the regular graph the floor has the same make-up but falls with size, so the largest lives stay
  noise-dominated ($1-w$ small). On the power-law graph the larger firms are better connected: a larger field amplitude keeps
  their core up and more frequent killing dips shorten their lives, which raises the entry and exit term. In the lognormal
  formula of notebook 05, $s^2\simeq(1-w)^2\operatorname{Var}(\ln V_F\mid\bar S)$, the degree acts mostly through $(1-w)^2$ and
  the life length through the variance.

**The transient terms.** For a logistic climb from $S_0$ to the plateau $h$ in a constant field, and a decline in a field that
has fallen to $-\varphi$ until the size $S_{\rm end}$, integrating $(d\ln S/dt)^2$ gives

$$\sum_{\rm entry}g^2\simeq\Delta t\,h\Big(\ln\frac h{S_0}-1+\frac{S_0}h\Big),\qquad
\sum_{\rm exit}g^2\simeq\Delta t\Big(\varphi\ln\frac h{S_{\rm end}}+h-S_{\rm end}\Big).$$

In [4]:
d = floor_parts("powerlaw")
sel = d["complete"] == 1
Sb, h, x0, Se, phi = (d[k][sel] for k in ("Sbar", "h", "x0", "Send", "phi"))
pin = eco.DT * h * (np.log(h / x0) - 1 + x0 / h)
pout = eco.DT * (phi * np.log(h / Se) + h - Se)
P = np.array_split(np.argsort(Sb), 5)
print(f"{sel.sum()} complete lives (born and exited inside the window)")
print(" Sbar  plateau | entry: measured  formula  log-corr | exit: measured  formula  log-corr")
for p in P:
    a, b = d["SSent"][sel][p], d["SSext"][sel][p]
    ok, okx = (a > 0) & (pin[p] > 0), (b > 0) & (pout[p] > 0)
    print(f"{Sb[p].mean():5.2f}  {h[p].mean():5.2f}  |      {a.mean():5.2f}     {pin[p].mean():5.2f}     "
          f"{np.corrcoef(np.log(a[ok]), np.log(pin[p][ok]))[0, 1]:.2f}   |     {b.mean():5.2f}    {pout[p].mean():5.2f}     "
          f"{np.corrcoef(np.log(b[okx]), np.log(pout[p][okx]))[0, 1]:.2f}")

175171 complete lives (born and exited inside the window)
 Sbar  plateau | entry: measured  formula  log-corr | exit: measured  formula  log-corr
 0.34   0.41  |       0.33      0.18     0.52   |      0.96     1.34     0.91
 0.67   0.87  |       0.84      0.58     0.54   |      1.36     1.86     0.90
 1.00   1.31  |       1.30      1.10     0.53   |      1.70     2.20     0.89
 1.44   1.85  |       1.81      1.82     0.51   |      2.09     2.51     0.87
 2.59   3.19  |       2.94      4.07     0.57   |      3.03     3.22     0.85


The exit formula follows the measured decline life by life; the entry formula has the right scale but correlates
less, because a firm climbs in a field that moves. Both grow with the plateau, and both are divided by the life length.

## How long a firm lives

An established firm dies in a dip of its field. Inside a dip the firm is small, the logistic term is negligible, and the
firm is a Feller branching diffusion $dS=\Phi(t)S\,dt+\sqrt{DS}\,dW$, whose extinction probability is exact:

$$P_{\rm kill}=\exp\Big(-\frac2{D\,J}\Big),\qquad J=\int\frac{dt}{S_{\rm det}(t)},$$

with $S_{\rm det}$ the noise-free path through the dip. At the bottom of the dip, $J\simeq\sqrt{2\pi/\Phi'}/S_{\min}$: the dip
kills when its area brings the firm below a size of order $D$ times its recovery time. This replaces the kill rule of
the deterministic model (thesis), where the listing threshold played the role of $D$. Test: for every downcrossing of zero by the field of a node
whose firm is established ($S\ge0.3$), restart the noise-free path at the firm's actual size, integrate until it recovers to
that size (at most 30 time units), and compare $P_{\rm kill}$ with whether the firm died.

In [5]:
def dips(kind="powerlaw", horizon=60, dt=0.01):
    path = FLOOR / f"dips_{kind}.npz"
    if not path.exists():
        per, rows = int(round(eco.DT / dt)), []
        for seed in eco.SEEDS:
            S, L, deg, _, Phi = eco.reference_economy(seed, kind)
            N, T = S.shape
            ii, kk = np.nonzero((Phi[:, :-1] > 0) & (Phi[:, 1:] <= 0))
            ok = (kk < T - horizon) & (S[ii, kk] >= 0.3)
            ii, kk = ii[ok], kk[ok]
            for c0 in range(0, ii.size, 40000):
                i, k = ii[c0:c0 + 40000], kk[c0:c0 + 40000]
                ph = Phi[i[:, None], k[:, None] + np.arange(horizon + 1)[None, :]]
                s0 = S[i, k]
                lnS, J = np.log(s0), np.zeros(i.size)
                done, trec = np.zeros(i.size, bool), np.full(i.size, horizon)
                with np.errstate(over="ignore", divide="ignore"):
                    for q in range(horizon):
                        p0, p1 = ph[:, q], ph[:, q + 1]
                        for j in range(per):
                            Sd = np.exp(lnS)
                            J += np.where(done, 0.0, dt / Sd)
                            lnS = lnS + (p0 + (p1 - p0) * j / per - Sd) * dt
                        rec = (~done) & (np.exp(lnS) >= s0) & (p1 > 0)
                        trec, done = np.where(rec, q + 1, trec), done | rec
                died = L[i, np.minimum(k + trec, T - 1)] != L[i, k]
                rows.append(np.c_[J, died, deg[i] / deg.mean(), s0])
        np.savez(path, rows=np.concatenate(rows))
    return np.load(path)["rows"]


J, died = dips()[:, 0], dips()[:, 1]
with np.errstate(over="ignore", divide="ignore"):
    p = np.exp(-2 / (eco.D * np.nan_to_num(J, posinf=1e300)))
print(f"{J.size} dips of established firms: died {died.mean():.3f}, predicted {p.mean():.3f}")
print("  predicted P_kill     dips    predicted  observed")
for a, b in zip([0, 1e-3, .01, .05, .1, .2, .3, .4, .5, .6, .7, .8, .9, .95, .99], [1e-3, .01, .05, .1, .2, .3, .4, .5, .6, .7, .8, .9, .95, .99, 1.01]):
    s = (p >= a) & (p < b)
    print(f"  {a:5.3f}-{min(b, 1):5.3f}    {s.sum():7d}     {p[s].mean():.3f}     {died[s].mean():.3f}")
ll = lambda q: np.mean(died * np.log(np.clip(q, 1e-9, 1)) + (1 - died) * np.log(np.clip(1 - q, 1e-9, 1)))
print(f"log-likelihood per dip: Feller {ll(p):.3f}, constant rate {ll(np.full_like(p, died.mean())):.3f}")

446554 dips of established firms: died 0.722, predicted 0.725
  predicted P_kill     dips    predicted  observed
  0.000-0.001       5990     0.000     0.000
  0.001-0.010      12094     0.005     0.003
  0.010-0.050      21758     0.027     0.015
  0.050-0.100      15973     0.073     0.043
  0.100-0.200      22038     0.147     0.102
  0.200-0.300      17048     0.248     0.200
  0.300-0.400      14987     0.349     0.316
  0.400-0.500      13860     0.450     0.418
  0.500-0.600      13560     0.550     0.540
  0.600-0.700      13909     0.651     0.662
  0.700-0.800      15461     0.751     0.780
  0.800-0.900      20143     0.853     0.894
  0.900-0.950      15241     0.927     0.954
  0.950-0.990      25475     0.974     0.986
  0.990-1.000     219017     0.999     1.000
log-likelihood per dip: Feller -0.175, constant rate -0.591


The kill law is calibrated over the whole range with no fitted parameter, slightly too soft at the two ends.

**Rice rate and memoryless lives.** Downcrossings of zero by a Gaussian field of mean $m$, standard deviation $f$ and
autocorrelation $\rho$ occur at the rate $\nu=\sqrt{-\rho''(0)}\,e^{-u^2/2}/2\pi$, $u=m/f$. Per node, with $\rho''(0)$ from the
lag-$\Delta t$ autocorrelation, against the observed rate, and the life lengths of established, complete lives:

In [6]:
def node_fields(kind="powerlaw"):
    path = FLOOR / f"nodes_{kind}.npz"
    if not path.exists():
        rows, lvs = [], []
        for seed in eco.SEEDS:
            S, L, deg, _, Phi = eco.reference_economy(seed, kind)
            N, T = S.shape
            m, f = Phi.mean(1), Phi.std(1)
            x = Phi - m[:, None]
            rho1 = np.sum(x[:, 1:] * x[:, :-1], 1) / np.sum(x * x, 1)
            dc = np.sum((Phi[:, :-1] > 0) & (Phi[:, 1:] <= 0), 1) / (T * eco.DT)
            rows.append(np.c_[deg / deg.mean(), m, f, rho1, dc])
            for i in range(N):
                for seg in np.split(np.arange(T), np.flatnonzero(np.diff(L[i])) + 1):
                    if seg[0] > 0 and seg[-1] < T - 1 and seg.size >= 21:
                        lvs.append((seed * N + i, seg.size - 1))
        np.savez(path, nodes=np.concatenate(rows), lives=np.array(lvs))
    z = np.load(path)
    return z["nodes"], z["lives"]


nodes, lv = node_fields()
kap, m, f, rho1, dc = nodes.T
u = m / f
rice = np.sqrt(2 * (1 - rho1) / eco.DT ** 2) / (2 * np.pi) * np.exp(-u ** 2 / 2)
ub = np.quantile(u, np.linspace(0, 1, 11))
lu = u[lv[:, 0].astype(int)]
print("      u        kappa   downcrossings/unit time: observed  Rice  | established lives: mean n  CV of n - 20")
for a, b in zip(ub[:-1], ub[1:]):
    s, sl = (u >= a) & (u < b), (lu >= a) & (lu < b)
    nn = lv[sl, 1]
    print(f"  {a:+5.2f}..{b:+5.2f}   {kap[s].mean():4.2f}            {dc[s].mean():.4f}      {rice[s].mean():.4f} |"
          f"     {nn.mean():6.1f}      {(nn - 20).std() / (nn - 20).mean():.2f}")

      u        kappa   downcrossings/unit time: observed  Rice  | established lives: mean n  CV of n - 20
  -10.64..-1.78   2.43            0.0071      0.0088 |       23.0      1.22
  -1.78..-1.18   1.21            0.0338      0.0383 |       25.0      1.10
  -1.18..-0.76   1.01            0.0626      0.0673 |       27.4      1.14
  -0.76..-0.38   0.92            0.0841      0.0898 |       31.4      1.15
  -0.38..-0.04   0.84            0.0938      0.1024 |       36.6      1.17
  -0.04..+0.30   0.81            0.0932      0.1034 |       43.2      1.18
  +0.30..+0.67   0.76            0.0849      0.0939 |       53.6      1.16
  +0.67..+1.10   0.72            0.0662      0.0726 |       74.5      1.11
  +1.10..+1.71   0.67            0.0394      0.0427 |      127.6      1.05
  +1.71..+6.62   0.62            0.0106      0.0107 |      222.4      0.91


Rice's formula gives the downcrossing rate from the node's own $(m,f,\rho)$ to within 1–13% in every decile of $u$ but the
lowest (the hubs, 24%); it always lies above, partly because sampling every $\Delta t$ misses brief crossings. Established lives are
memoryless (coefficient of variation of $n-20$ near one; below one at the highest $u$, where the window
cuts the longest lives). The hazard of an established firm is the downcrossing rate times the mean $P_{\rm kill}$ of its dips,
so the mean life grows steeply with $u$, and with it the spread of $\ln n$ among the largest firms: small firms live on nodes
whose lives barely exceed the cut $n\ge20$, large firms on nodes with long exponential lives.

## What the field must provide

The one-firm reduction of notebook 04 uses each node's measured field. Replace it by a Gaussian field
$\Phi_i(t)=m_i+f_i\eta_i(t)$, with $\eta$ of unit variance and the network-mean autocorrelation (circulant embedding), and
simplify the node statistics step by step: the amplitude at its pure $\sqrt\kappa$ law, the means reshuffled within degree
classes, the means at their class mean, and finally the cavity law with four constants,
$m\mid\kappa\sim\mathcal N(a-b\kappa,\,s^2\kappa)$, $f=c\sqrt\kappa$, fitted to the measured fields.

**The field's statistics, as the paper quotes them.** Per degree class of `scripts/two_time_dmft.py` (classes that resolve
the hubs), the temporal variance of the field and the frozen spread of its time means, less the part due to the spread of
degrees inside the class, each over $\kappa$ and averaged in logs with the class weights; and the class-weighted
correlation of the moving part. These are the measured inputs of the closure tests in the paper's appendix. The shape of
the frozen spread is measured on the residuals of the time means about a linear trend in the node's own degree inside its
class, divided by $\sqrt r$ ($r$ the degree over the class mean) and standardised in each class.

In [ ]:
import sys
sys.path.insert(0, str(eco.ROOT / "scripts"))
import two_time_dmft as tt


def field_nodes(kind="powerlaw", nlag=21):
    """Per node of each reference economy: degree, time-mean field m, temporal sd f, mean size, and the autocovariance
    of the field about its time mean at lags 0..nlag-1 samples."""
    path = FLOOR / f"field_nodes_{kind}.npz"
    if not path.exists():
        out = {}
        for seed in eco.SEEDS:
            S, L, deg, _, Phi = eco.reference_economy(seed, kind)
            x = Phi - Phi.mean(1, keepdims=True)
            T = x.shape[1]
            acov = np.stack([np.sum(x[:, l:] * x[:, :T - l], 1) / (T - l) for l in range(nlag)], 1)
            out[f"seed{seed}"] = np.c_[deg, Phi.mean(1), Phi.std(1), S.mean(1), acov]
        np.savez(path, **out)
    z = np.load(path)
    return [z[f"seed{s}"] for s in eco.SEEDS]


def within_class(deg, m):
    """Residuals of the time-mean field about a linear trend in the node's own degree inside each degree class of
    scripts/two_time_dmft.py (classes that resolve the hubs), over sqrt(r), r the degree over the class mean,
    standardised in each class.  Returns (trend, sd, standardised residual) per node, and the class lists."""
    o, _, _ = tt.degree_classes(deg)
    trend, sd, z = np.empty(deg.size), np.empty(deg.size), np.empty(deg.size)
    for i in o:
        r = deg[i] / deg[i].mean()
        X = np.c_[np.ones(i.size), r] if r.std() > 1e-9 else np.ones((i.size, 1))
        trend[i] = X @ np.linalg.lstsq(X, m[i], rcond=None)[0]
        e = (m[i] - trend[i]) / np.sqrt(r)
        sd[i], z[i] = e.std(), (e - e.mean()) / e.std()
    return trend, sd, z, o


nodes = field_nodes()
# the three constants of the closure tests, as scripts/two_time_dmft.measured_temporal computes them
order, w, Kc = tt.degree_classes(nodes[0][:, 0])
kappa = Kc.sum(1)
cl = []
for X in nodes:
    o, _, _ = tt.degree_classes(X[:, 0])
    cl.append(dict(frozen=np.array([X[i, 1].var() for i in o]), moving=np.array([np.mean(X[i, 2] ** 2) for i in o]),
                   ES=np.array([X[i, 3].mean() for i in o]), acov=np.array([X[i, 4:].mean(0) for i in o])))
sim = {k: np.mean([c_[k] for c_ in cl], 0) for k in cl[0]}
c_mov = np.exp(w @ (0.5 * np.log(sim["moving"] / kappa)))
var_r = np.array([(nodes[0][i, 0] / nodes[0][i, 0].mean()).var() for i in order])
frz = sim["frozen"] - (eco.MU * (Kc @ sim["ES"])) ** 2 * var_r       # less the spread of degrees within a class
ok = frz > 0
f_frz = np.exp(w[ok] @ (0.5 * np.log(frz[ok] / kappa[ok])) / w[ok].sum())
rho = (w @ sim["acov"]) / (w @ sim["acov"][:, 0])
lag = np.arange(0, 10, 0.1)
tau = lag[np.argmax(np.interp(lag, eco.DT * np.arange(rho.size), rho) < np.exp(-1))]
K = np.concatenate([X[:, 0] / X[:, 0].mean() for X in nodes])
Fv, Mv = np.concatenate([X[:, 2] for X in nodes]), np.concatenate([X[:, 1] for X in nodes])
print(f"temporal amplitude {c_mov:.3f} sqrt(kappa), frozen spread at equal degree {f_frz:.3f} sqrt(kappa) "
      f"({len(w)} classes, {(~ok).sum()} dropped), 1/e time of the class-weighted correlation {tau:.1f}")
print(f"amplitude per node f ~ kappa^{np.polyfit(np.log(K), np.log(Fv), 1)[0]:.2f} (fit over nodes)")
print("\nshape of the spread of mean fields at equal degree (within-class residuals, standardised):")
for s, X in zip(eco.SEEDS, nodes):
    _, _, z, o = within_class(X[:, 0], X[:, 1])
    kc = np.array([stats.kurtosis(z[i]) for i in o])
    print(f"  seed {s}: skewness {stats.skew(z):+.2f}, excess kurtosis {stats.kurtosis(z):+.2f} "
          f"(classes: median {np.median(kc):+.2f}, range {kc.min():+.1f}..{kc.max():+.1f})")
# notebook 04's first estimate pooled 20 quantile classes of both seeds and standardised the raw means in each
edges = np.quantile(K, np.linspace(0, 1, 21))
b = np.clip(np.digitize(K, edges[1:-1]), 0, 19)
zc = [(Mv[b == j] - Mv[b == j].mean()) / Mv[b == j].std() for j in range(20)]
print(f"  raw means in 20 quantile classes, seeds pooled: excess kurtosis {stats.kurtosis(np.concatenate(zc)):+.2f}; "
      f"top class alone (kappa {K[b == 19].min():.1f}..{K[b == 19].max():.0f}) {stats.kurtosis(zc[19]):+.1f}, "
      f"the other 19 {stats.kurtosis(np.concatenate(zc[:19])):+.2f}")

The spread of the mean field at equal degree is **close to Gaussian**: excess kurtosis about $0.3$ on both economies, and no
class far from it. Pooling raw quantile classes gives about $2$, but only through the top class, whose degrees span more
than a decade, so that the fall of the mean field with the degree reads as a fat tail. The first version of notebook 04,
and of the paper, quoted that value.

In [7]:
MAXLAG = 120


def gauss_paths(acf, n, T, rng):
    """Stationary unit-variance Gaussian paths with autocorrelation acf (zero beyond its length), circulant embedding."""
    M = 2 * (T + len(acf))
    r = np.zeros(M); r[:len(acf)] = acf; r[M - len(acf) + 1:] = acf[1:][::-1]
    lam = np.clip(np.fft.rfft(r).real, 0, None)
    out = np.empty((n, T))
    for c0 in range(0, n, 4000):
        k = min(4000, n - c0)
        out[c0:c0 + k] = np.fft.irfft(np.fft.rfft(rng.standard_normal((k, M))) * np.sqrt(lam), n=M)[:, :T]
    return out / out.std()


def one_firm(Phi, S0, rng, dt=0.01):
    """The one-firm equation with demographic noise and turnover along given fields (linear between samples)."""
    N, T = Phi.shape
    per = int(round(eco.DT / dt))
    S, life = S0.copy(), np.zeros(N, np.int64)
    oS, oL = np.empty((N, T)), np.empty((N, T), np.int64)
    oS[:, 0], oL[:, 0] = S, life
    for k in range(T - 1):
        for j in range(per):
            S = S + S * (Phi[:, k] + (Phi[:, k + 1] - Phi[:, k]) * j / per - S) * dt \
                + np.sqrt(eco.D * np.maximum(S, 0) * dt) * rng.standard_normal(N)
            dead = S <= 0
            if dead.any():
                S[dead] = eco.D; life[dead] += 1
        oS[:, k + 1], oL[:, k + 1] = S, life
    return oS, oL


def mean_acf(x, T):
    ac = np.array([np.mean(np.sum(x[:, l:] * x[:, :T - l], 1)) / T for l in range(MAXLAG)])
    ac /= ac[0]
    if (ac < 0).any():
        ac[np.argmax(ac < 0):] = 0
    return ac


VARIANTS = ("measured field", "Gaussian, node (m, f)", "f = c sqrt(kappa), m shuffled in class",
            "m at class mean", "cavity law, four constants")


def surrogate_panels(kind="powerlaw"):
    path = FLOOR / f"surrogates_{kind}.npz"
    if not path.exists():
        out = {}
        for vi, variant in enumerate(VARIANTS):
            rec = []
            for seed in eco.SEEDS:
                S, L, deg, _, Phi = eco.reference_economy(seed, kind)
                N, T = S.shape
                rng = np.random.default_rng(900 + 10 * vi + seed)
                m, f, kap = Phi.mean(1), Phi.std(1), deg / deg.mean()
                ac = mean_acf((Phi - m[:, None]) / f[:, None], T)
                cls = np.array_split(np.argsort(kap), 40)
                c = np.exp(np.mean(np.log(f) - 0.5 * np.log(kap)))
                if variant == "measured field":
                    F = Phi
                else:
                    mm, ff = m.copy(), f.copy()
                    if variant.startswith("f = c"):
                        ff = c * np.sqrt(kap)
                        for cc in cls: mm[cc] = rng.permutation(m[cc])
                    elif variant == "m at class mean":
                        ff = c * np.sqrt(kap)
                        for cc in cls: mm[cc] = m[cc].mean()
                    elif variant.startswith("cavity"):
                        (a0, a1), *_ = np.linalg.lstsq(np.c_[np.ones(N), kap], m, rcond=None)
                        s2 = np.mean((m - a0 - a1 * kap) ** 2 / kap)
                        mm, ff = a0 + a1 * kap + np.sqrt(s2 * kap) * rng.standard_normal(N), c * np.sqrt(kap)
                    F = mm[:, None] + ff[:, None] * gauss_paths(ac, N, T, rng)
                Ss, Ls = one_firm(F, S[:, 0], rng)
                rec += [(x[0], x[1], x[2]) for x in lives(Ss, Ls)]
            out[f"v{vi}"] = np.array(rec)
        np.savez(path, **out)
    return np.load(path)


def panel_line(rec, nm):
    s = rec[:, 2] >= nm
    z = zeta_bins(rec[s, 0], rec[s, 1])[0]
    return z


full = []
for seed in eco.SEEDS:
    d_ = np.load(eco.simulate_reference(seed))
    full += [(x[0], x[1], x[2]) for x in lives(d_["S"].astype(float), d_["L"])]
full = np.array(full)
sp = surrogate_panels()
print(f"{'':40s} n >= 20: zeta_1  ratios            n >= 5: zeta_1  ratios")
for name, rec in [("full model", full)] + [(v, sp[f"v{i}"]) for i, v in enumerate(VARIANTS)]:
    z20, z5 = panel_line(rec, 20), panel_line(rec, 5)
    print(f"{name:40s}  {z20[0]:.3f}  {f2(z20[1:] / z20[0])}          {z5[0]:.3f}  {f2(z5[1:] / z5[0])}")

                                         n >= 20: zeta_1  ratios            n >= 5: zeta_1  ratios
full model                                0.253   1.79  2.37  2.80          0.208   1.94  2.75  3.41


measured field                            0.262   1.79  2.39  2.83          0.213   1.94  2.75  3.42
Gaussian, node (m, f)                     0.231   1.78  2.37  2.81          0.194   1.95  2.81  3.55
f = c sqrt(kappa), m shuffled in class    0.277   1.81  2.42  2.86          0.209   1.96  2.82  3.53


m at class mean                           0.221   1.93  2.79  3.56          0.193   2.06  3.12  4.14
cavity law, four constants                0.293   1.83  2.50  3.02          0.214   2.00  2.94  3.80


- **Three numbers per node suffice.** A Gaussian field with the node's mean and amplitude and one correlation function for
  every node reproduces the full model; neither class-specific spectra nor the fat tails of the dips matter.
- **The amplitude spread at fixed degree seems not to matter here, but only by compensation** (the spread tests below):
  the 40 quantile classes put hubs of very different degree in the top class, and reshuffling the means there adds bend,
  while $f=c\sqrt\kappa$ takes some away.
- **The spread of the mean within a class is essential.** It decides how high a firm's plateau is and how long it lives;
  without it the bend weakens.
- **The four-constant cavity law** keeps most of the bend for long lives and raises $\zeta_1$ a little; for short lives
  ($n\ge5$) its ratios move towards $q$. What it misses is the spread of the amplitudes at equal degree, not a fat tail of the
  means, which are close to Gaussian inside a class (above).

The four constants are the order parameters of the mean-field theory: $s^2=\sigma^2Q_T$, $c^2=\sigma^2(Q_0-Q_T)$, $b=\mu\bar B$,
$a=1-\bar f$, with $Q_0$, $Q_T$ and $\bar B$ degree-weighted moments of the sizes. Measured on the fields against the same
quantities measured on the sizes alone:

**Which spread matters.** Inside the degree classes of the closure, keep each node's own mean field and amplitude and
change one thing at a time, with the same temporal and demographic noise in every variant: the residual of the means about
the within-class trend replaced by a Gaussian of the class's variance; the amplitudes (over $\sqrt\kappa$) reshuffled among
the nodes of the class; set to the class's root mean square; or set to the closure's single law $c\sqrt\kappa$; and, last, both at their class law (the means at the within-class
trend), which removes every difference between nodes of equal degree.

In [ ]:
SPREAD = ("own m, own f", "own m, Gaussian residual", "own m, f shuffled in class", "own m, f at class law",
          "own m, f = c sqrt(kappa)", "m and f at class law")


def spread_panels(kind="powerlaw"):
    """The Gaussian-field surrogate with the same temporal and demographic noise in every variant, inside the degree
    classes of scripts/two_time_dmft.py: each node's own (m, f); the means' residual about the within-class trend
    replaced by a Gaussian of the class's variance; the amplitudes (over sqrt(kappa)) reshuffled within the class;
    set to the class's root mean square; the single law c sqrt(kappa) of the closure; and both the means (at their
    within-class trend) and the amplitudes at their class law."""
    path = FLOOR / f"spread_{kind}.npz"
    if not path.exists():
        out = {v: [] for v in SPREAD}
        for seed in eco.SEEDS:
            S, L, deg, _, Phi = eco.reference_economy(seed, kind)
            N, T = S.shape
            m, f, kap = Phi.mean(1), Phi.std(1), deg / deg.mean()
            ac = mean_acf((Phi - m[:, None]) / f[:, None], T)
            del Phi
            trend, sd, _, o = within_class(deg.astype(float), m)
            r = np.empty(N)
            for i in o:
                r[i] = deg[i] / deg[i].mean()
            fr, eta = f / np.sqrt(kap), gauss_paths(ac, N, T, np.random.default_rng(55 + seed))
            c = np.exp(np.mean(np.log(fr)))
            for vi, v in enumerate(SPREAD):
                rng = np.random.default_rng(700 + 10 * vi + seed)
                mm, ff = m, f
                if v == "own m, Gaussian residual":
                    mm = trend + np.sqrt(r) * sd * rng.standard_normal(N)
                elif v == "own m, f shuffled in class":
                    ff = np.empty(N)
                    for i in o:
                        ff[i] = rng.permutation(fr[i]) * np.sqrt(kap[i])
                elif v in ("own m, f at class law", "m and f at class law"):
                    ff = np.empty(N)
                    for i in o:
                        ff[i] = np.sqrt(np.mean(fr[i] ** 2)) * np.sqrt(kap[i])
                elif v == "own m, f = c sqrt(kappa)":
                    ff = c * np.sqrt(kap)
                if v == "m and f at class law":
                    mm = trend
                Ss, Ls = one_firm(mm[:, None] + ff[:, None] * eta, S[:, 0], np.random.default_rng(77 + seed))
                out[v] += [(x[0], x[1], x[2]) for x in lives(Ss, Ls)]
        np.savez(path, **{f"v{i}": np.array(out[v]) for i, v in enumerate(SPREAD)})
    return np.load(path)


sp2 = spread_panels()
print(f"{'':30s} n >= 20: zeta_1  ratios            n >= 5: zeta_1  ratios")
for name, rec in [("full model", full)] + [(v, sp2[f"v{i}"]) for i, v in enumerate(SPREAD)]:
    z20, z5 = panel_line(rec, 20), panel_line(rec, 5)
    print(f"{name:30s}  {z20[0]:.3f}  {f2(z20[1:] / z20[0])}          {z5[0]:.3f}  {f2(z5[1:] / z5[0])}")

- **The shape of the spread of the means does not matter**: a Gaussian residual gives the same panel.
- **The spread of the amplitudes at equal degree does.** Reshuffling them inside the class changes nothing, so their
  correlation with the mean is irrelevant, but setting them to the class value raises $\zeta_1$ by about $0.05$ and moves the
  ratios towards $q$, as much as $c\sqrt\kappa$ and as far as the two-time closure fed the three measured constants (paper,
  closure appendix): that closure replaces this spread by its mean.
- **Without either spread the bend is gone** (last line).
- The 40-class reshuffle of the surrogate above agrees with the full model only by compensation.

In [8]:
for seed in eco.SEEDS:
    S, L, deg, a, Phi = eco.reference_economy(seed)
    N = S.shape[0]
    kap, w = deg / deg.mean(), deg / deg.sum()
    m, f = Phi.mean(1), Phi.std(1)
    (a0, a1), *_ = np.linalg.lstsq(np.c_[np.ones(N), kap], m, rcond=None)
    Sb = S.mean(1)
    Bbar, Q0, QT = w @ Sb, w @ (S ** 2).mean(1), w @ Sb ** 2
    print(f"seed {seed}:  Var(m)/kappa {np.mean((m - a0 - a1 * kap) ** 2 / kap):.2f} vs sigma^2 Q_T {eco.SIGMA ** 2 * QT:.2f};   "
          f"<f^2/kappa> {np.mean(f ** 2 / kap):.2f} vs sigma^2 (Q_0 - Q_T) {eco.SIGMA ** 2 * (Q0 - QT):.2f};   "
          f"slope {-a1:.2f} vs mu Bbar {eco.MU * Bbar:.2f}")

seed 0:  Var(m)/kappa 5.95 vs sigma^2 Q_T 6.12;   <f^2/kappa> 4.57 vs sigma^2 (Q_0 - Q_T) 4.39;   slope 1.65 vs mu Bbar 1.49


seed 1:  Var(m)/kappa 8.17 vs sigma^2 Q_T 8.04;   <f^2/kappa> 4.67 vs sigma^2 (Q_0 - Q_T) 4.57;   slope 1.83 vs mu Bbar 1.59


The static variance and the amplitude follow the mean-field identities to a few per cent; the slope is 10–15% above
$\mu\bar B$ because the bilinear kernel misses the structural cutoff of the hubs (the soft-configuration kernel below accounts
for it).

## Closing the chain: two-time mean-field theory

The Gaussian-field surrogate is the single-site process of heterogeneous dynamical mean-field theory. Close it
self-consistently: each node, with its own degree and its hidden degree from the soft-configuration kernel $K$ of notebook 02,
sees

$$\Phi_i(t)=a-\mu\sum_rK_{ir}M_r+\sigma\Big(\sum_rK_{ir}Q_{T,r}\Big)^{1/2}z_i+\sigma\Big(\sum_rK_{ir}\big[C_r(0)-Q_{T,r}\big]\Big)^{1/2}\eta_i(t),$$

where $M_r$, $Q_{T,r}$ (mean squared time-average) and $C_r(L)$ (two-time size correlation) are measured on the single sites of
degree class $r$, $\eta_i$ has the class's kernel-weighted correlation, and $a$ keeps the mean size at one. Simulate the single
sites with noise and turnover, measure, update with damping, repeat. Start from a neutral guess, not from the network: the
fixed point is a prediction.

In [9]:
def hdmft_sampler(kind="powerlaw", iters=26, ncl=24, burn=200, tobs=480, damp=0.5, gain=1.2):
    path = FLOOR / f"hdmft_{kind}.npz"
    if path.exists():
        return np.load(path)
    k = np.diff(eco.reference_coupling(0, kind).indptr).astype(float)
    N = k.size
    if kind == "regular":
        Kn, cls = np.ones((N, 1)), [np.arange(N)]
    else:
        from scipy.optimize import least_squares
        chunks = np.array_split(np.sort(k), ncl)
        rep, wts, conn = np.array([c.mean() for c in chunks]), np.array([c.size for c in chunks]) / N, k.mean() / N
        frac = rep / N
        kern = lambda lh: (lambda pr: pr / (pr + conn))(np.exp(lh)[:, None] * np.exp(lh)[None, :])
        sol = least_squares(lambda lh: (kern(lh) @ wts - frac) / frac, np.log(frac), bounds=(-50, 20),
                            xtol=1e-14, ftol=1e-14, gtol=1e-14, max_nfev=20000)
        hcl = np.exp(sol.x)
        hn = np.exp(np.interp(np.log(k), np.log(rep), np.log(hcl)))
        Kn = hn[:, None] * hcl[None, :] / (hn[:, None] * hcl[None, :] + conn) * wts[None, :] / conn
        Kn *= (k / k.mean() / Kn.sum(1))[:, None]
        cls = np.array_split(np.argsort(k), ncl)
    R, wk = len(cls), np.array([k[c].sum() for c in cls])
    M, QT = np.ones(R), np.ones(R)
    Qd = np.outer(np.ones(R), 0.8 * np.exp(-np.arange(MAXLAG) * eco.DT / 3.0))
    a, S0, log = 1.0, np.ones(N), []
    for it in range(iters + 1):
        final = it == iters
        T = burn + (2 * tobs + 1 if final else tobs)
        rng = np.random.default_rng(1000 + it)
        acfs = []
        for r in range(R):
            x = Kn[cls[r]].mean(0) @ Qd; x = x / x[0]
            if (x < 0).any(): x[np.argmax(x < 0):] = 0
            acfs.append(x)
        eta = np.empty((N, T))
        for r in range(R):
            eta[cls[r]] = gauss_paths(acfs[r], cls[r].size, T, rng)
        mean = a - eco.MU * Kn @ M + eco.SIGMA * np.sqrt(np.maximum(Kn @ QT, 0)) * rng.standard_normal(N)
        Phi = mean[:, None] + eco.SIGMA * np.sqrt(np.maximum(Kn @ Qd[:, 0], 1e-9))[:, None] * eta
        S, L = one_firm(Phi, S0, rng)
        So = S[:, burn:]
        F = np.fft.rfft(So, n=2 * So.shape[1])
        ac = np.fft.irfft(F * np.conj(F), n=2 * So.shape[1])[:, :MAXLAG] / (So.shape[1] - np.arange(MAXLAG))[None, :]
        Sb = So.mean(1)
        Mn = np.array([So[c].mean() for c in cls])
        QTn = np.array([np.mean(Sb[c] ** 2) for c in cls])
        Cn = np.array([ac[c].mean(0) for c in cls])
        log.append((it, a, So.mean(), wk @ Mn / wk.sum(), wk @ QTn / wk.sum(), wk @ (Cn[:, 0] - QTn) / wk.sum(),
                    (L[:, -1] - L[:, burn]).mean() / ((T - burn) * eco.DT)))
        if final:
            np.savez(path, S=So.astype(np.float32), L=L[:, burn:], log=np.array(log), a=a)
            return np.load(path)
        a += gain * (1 - So.mean())
        M, QT = (1 - damp) * M + damp * Mn, (1 - damp) * QT + damp * QTn
        Qd = (1 - damp) * Qd + damp * (Cn - QTn[:, None])
        S0 = S[:, -1]


def full_order_parameters(kind="powerlaw"):
    out = []
    for seed in eco.SEEDS:
        d_ = np.load(eco.simulate_reference(seed, kind))
        S, L, k = d_["S"].astype(float), d_["L"], d_["deg"].astype(float)
        w, Sb, T = k / k.sum(), S.mean(1), S.shape[1]
        out.append((w @ Sb, w @ Sb ** 2, w @ (S ** 2).mean(1) - w @ Sb ** 2, (L[:, -1] - L[:, 0]).mean() / ((T - 1) * eco.DT)))
    return np.array(out)


H = hdmft_sampler()
print(" it      a    <S>   Bbar    Q_T   Q_0-Q_T  exits")
for r in H["log"][::5].tolist() + [H["log"][-1].tolist()]:
    print(f"{int(r[0]):3d}  {r[1]:5.3f}  {r[2]:5.3f}  {r[3]:5.3f}  {r[4]:5.3f}   {r[5]:5.3f}  {r[6]:.3f}")
fo = full_order_parameters()
print("full model, two seeds:  Bbar " + " ".join(f"{x:.3f}" for x in fo[:, 0]) + "   Q_T " + " ".join(f"{x:.3f}" for x in fo[:, 1])
      + "   Q_0-Q_T " + " ".join(f"{x:.3f}" for x in fo[:, 2]) + "   exits " + " ".join(f"{x:.3f}" for x in fo[:, 3]))
rec = np.array([(x[0], x[1], x[2]) for x in lives(H["S"].astype(float), H["L"])])
for name, r_ in (("full model", full), ("two-time theory", rec)):
    z20, z5 = panel_line(r_, 20), panel_line(r_, 5)
    print(f"{name:16s} n >= 20: zeta_1 {z20[0]:.3f}, ratios {f2(z20[1:] / z20[0])}    n >= 5: zeta_1 {z5[0]:.3f}, ratios {f2(z5[1:] / z5[0])}")

 it      a    <S>   Bbar    Q_T   Q_0-Q_T  exits
  0  1.000  0.579  0.423  0.567   0.370  0.566
  5  1.436  1.047  0.871  2.256   0.655  0.430
 10  1.154  1.027  0.887  2.773   0.779  0.565
 15  1.023  1.020  0.925  3.197   0.947  0.633
 20  1.021  0.995  0.880  2.714   0.985  0.614
 25  1.017  0.984  0.877  2.628   1.064  0.597
 26  1.036  0.985  0.866  2.478   1.075  0.600


full model, two seeds:  Bbar 0.849 0.905   Q_T 2.116 2.784   Q_0-Q_T 1.519 1.581   exits 0.572 0.653


full model       n >= 20: zeta_1 0.253, ratios  1.79  2.37  2.80    n >= 5: zeta_1 0.208, ratios  1.94  2.75  3.41
two-time theory  n >= 20: zeta_1 0.306, ratios  1.82  2.46  2.97    n >= 5: zeta_1 0.227, ratios  2.01  3.01  3.97


The theory, solved from a neutral start and with no input from the network simulation, gives the mean neighbour size,
the quenched variance and the turnover within the range of the two simulated economies, and the life panel for long lives:
the bend for $n\ge20$ comes out of a single site in a self-consistent Gaussian field. Two gaps remain:

- **The field moves too little.** The dynamic size variance $Q_0-Q_T$ is about 30% below the full model's, so the field
  amplitude is too small, large firms are too calm and $\zeta_1$ is about $0.05$ too high. It was still drifting upward at the
  last iterations, and extending the correlation from 60 to 200 time units does not change it. In the finite economy 10–30%
  of the field's temporal variance is a collective mode shared by all nodes, which the infinite-$N$ theory omits; the regular
  graph, where that mode is small, is the control (next cell).
- **The bend is too weak.** The theory gives every node of a class the same amplitude; the measured amplitudes differ by
  about 30% at equal degree, and that spread carries part of the bend, most visibly for short lives (spread tests above).

In [10]:
def collective_share(kind):
    out = []
    for seed in eco.SEEDS:
        _, _, _, _, Phi = eco.reference_economy(seed, kind)
        x = Phi - Phi.mean(1, keepdims=True)
        g = x.mean(0)
        out.append(np.sum(((x @ g) / (g @ g)) ** 2 * g.var()) / x.var(1).sum())
    return np.array(out)


for kind in ("powerlaw", "regular"):
    H_ = hdmft_sampler(kind)
    fo_ = full_order_parameters(kind)
    th = H_["log"][-1]
    print(f"{kind:9s} Q_T: full {f2(fo_[:, 1])}, theory {th[4]:.2f};   Q_0-Q_T: full {f2(fo_[:, 2])}, theory {th[5]:.2f};   "
          f"exits: full {f2(fo_[:, 3])}, theory {th[6]:.2f};   collective share of the field's motion {f2(collective_share(kind))}")

powerlaw  Q_T: full  2.12  2.78, theory 2.48;   Q_0-Q_T: full  1.52  1.58, theory 1.08;   exits: full  0.57  0.65, theory 0.60;   collective share of the field's motion  0.30  0.10


regular   Q_T: full  3.33  3.36, theory 3.31;   Q_0-Q_T: full  0.61  0.59, theory 0.45;   exits: full  0.69  0.69, theory 0.69;   collective share of the field's motion  0.03  0.03


**The control does not settle the first gap.** On the regular graph the collective mode carries about 3% of the field's
motion (2% at $N=64000$, where the full model's order parameters are unchanged), yet the sampler does not converge: $Q_T$ and
the turnover match the full model, but $Q_0-Q_T$ rises to about $0.76$ and then drifts down past the full model's $0.6$ to
$0.45$ after 26 iterations. The likely cause is the sampler, not the theory: $Q_T$ is estimated from finite windows, which
count part of the slow motion from one life to the next as static, and each iteration then moves that variance from the
moving to the frozen part of the field. Until the sampler runs with windows much longer than the lifetimes, the dynamic
variance, and with it the size of the first gap and its cause, is open; the static order parameters, the turnover and the
long-life bend do not depend on it.

## Predictions for firm data

If the bend comes from the climb after entry and the collapse before exit, cutting those years out of every life
should weaken it. Cut each measured life of the full model at the first and last time its noise-free path reaches half its
median, and recompute the exponents with the paper's estimator.

In [11]:
def trimmed_panels(kind="powerlaw"):
    path = FLOOR / f"trimmed_{kind}.npz"
    if not path.exists():
        names = ("full", "no entry", "no exit", "plateau only")
        rec = {k: [] for k in names}
        for seed in eco.SEEDS:
            S, L, deg, _, Phi = eco.reference_economy(seed, kind)
            lv = life_segments(L, 2)
            lv.sort(key=lambda x: x[1].size)
            for c0 in range(0, len(lv), 40000):
                part = lv[c0:c0 + 40000]
                ph, x0 = np.zeros((len(part), part[-1][1].size)), np.empty(len(part))
                for k, (i, seg) in enumerate(part):
                    ph[k, :seg.size], ph[k, seg.size:], x0[k] = Phi[i, seg], Phi[i, seg[-1]], S[i, seg[0]]
                det = eco.noise_free(ph, np.log(x0))
                for k, (i, seg) in enumerate(part):
                    m = seg.size
                    sd = np.exp(det[k, :m])
                    up = np.flatnonzero(sd >= 0.5 * np.median(sd)); a, b = up[0], up[-1]
                    for name, (lo, hi) in zip(names, ((0, m), (a, m), (0, b + 1), (a, b + 1))):
                        x = S[i, seg[lo:hi]]
                        if x.size >= 3:
                            g = np.diff(np.log(x))
                            v = np.sqrt(np.pi / 2) * np.abs(g - g.mean()).mean()
                            if v > 0:
                                rec[name].append((x.mean(), v, g.size))
        np.savez(path, **{k.replace(" ", "_"): np.array(v) for k, v in rec.items()})
    return np.load(path)


tp = trimmed_panels()
for name in ("full", "no_entry", "no_exit", "plateau_only"):
    z20, z5 = panel_line(tp[name], 20), panel_line(tp[name], 5)
    print(f"{name:13s} n >= 20: zeta_1 {z20[0]:.3f}, ratios {f2(z20[1:] / z20[0])}    n >= 5: zeta_1 {z5[0]:.3f}, ratios {f2(z5[1:] / z5[0])}")

full          n >= 20: zeta_1 0.253, ratios  1.79  2.37  2.80    n >= 5: zeta_1 0.208, ratios  1.94  2.75  3.41
no_entry      n >= 20: zeta_1 0.295, ratios  1.85  2.56  3.13    n >= 5: zeta_1 0.226, ratios  1.92  2.71  3.34
no_exit       n >= 20: zeta_1 0.295, ratios  1.86  2.58  3.16    n >= 5: zeta_1 0.240, ratios  1.94  2.77  3.48


plateau_only  n >= 20: zeta_1 0.383, ratios  1.92  2.74  3.44    n >= 5: zeta_1 0.294, ratios  1.95  2.78  3.48


Keeping only the plateau of each life steepens the decline and moves the ratios towards $q$: the climb and the collapse
both flatten the decline, by adding volatility to large firms, and make most of the bend. On firm data the test is to trim
the years after entry and before exit, or to condition on age. Compustat firms enter at listing, already large, so the
climb from the newborn size has no counterpart there and the entry part is probably overstated in the model.

**The tests of the paper's Sec. "Predictions for firm data"**, with the panel's estimator on the two reference economies:
the dispersion of $\ln v$ at fixed size, the shares of it explained by the life length and the degree, the response to
trimming a fixed number of years after a birth and before an exit (where they lie inside the window, keeping lives that
still have at least 20 growth rates), and the ratios by minimum life length. Every life is in `economy.life_table`.

In [ ]:
def trimmed_years(kind="powerlaw", ks=(0, 1, 2, 3, 5), nmin=20):
    """Lives with the first k years after a birth and the last k before an exit removed (where the birth or the exit
    lies inside the window), kept if at least nmin growth rates remain: rows (k, Sbar, v), the panel's estimator."""
    path = FLOOR / f"trimmed_years_{kind}.npz"
    if not path.exists():
        rows = []
        for seed in eco.SEEDS:
            d = np.load(eco.simulate_reference(seed, kind))
            lnS, L = np.log(d["S"].astype(float)), d["L"]
            T = lnS.shape[1]
            for i, seg in life_segments(L, nmin):
                born, exits = seg[0] > 0, seg[-1] < T - 1
                for k in ks:
                    x = lnS[i, seg[k if born else 0:seg.size - (k if exits else 0)]]
                    if x.size > nmin:
                        g = np.diff(x)
                        v = np.sqrt(np.pi / 2) * np.abs(g - g.mean()).mean()
                        if v > 0:
                            rows.append((k, np.exp(x).mean(), v))
        np.savez(path, rows=np.array(rows))
    return np.load(path)["rows"]


t = eco.life_table()
lv = np.log(t["v"])
fifths = lambda f, P: [np.mean([f(p) for p in P[j:j + 5]]) for j in range(0, 25, 5)]
print("1. Dispersion at fixed size: within-bin Var ln v, averaged over the five bins of each fifth of size")
for nm in (20, 5):
    s = np.flatnonzero(t["n"] >= nm)
    P = [s[p] for p in np.array_split(np.argsort(t["Sbar"][s]), 25)]
    print(f"   n >= {nm:2d}: " + "  ".join(f"{x:.3f}" for x in fifths(lambda p: lv[p].var(), P)))
s = np.flatnonzero(t["n"] >= 20)
P = [s[p] for p in np.array_split(np.argsort(t["Sbar"][s]), 25)]
r2 = lambda a: fifths(lambda p: np.corrcoef(lv[p], np.log(a[p]))[0, 1] ** 2, P)
print(f"2. Share of Var ln v within bins (n >= 20), by fifth: life length {f2(r2(t['n']))}, degree {f2(r2(t['deg']))}")
tr = trimmed_years()
print("3. Trimming k years after a birth and before an exit (lives still with >= 20 growth rates):")
for k in (0, 1, 2, 3, 5):
    r = tr[tr[:, 0] == k]
    z = zeta_bins(r[:, 1], r[:, 2])[0]
    print(f"   k = {k}: {len(r):6d} lives, zeta_1 {z[0]:.3f}, zeta_4/zeta_1 {z[3] / z[0]:.2f}")
print("4. Ratios by minimum life length:")
for nm in (2, 5, 20):
    s = t["n"] >= nm
    z = zeta_bins(t["Sbar"][s], t["v"][s])[0]
    print(f"   n >= {nm:2d}: {s.sum():7d} lives, zeta_1 {z[0]:.3f}, ratios {f2(z[1:] / z[0])}")

- **The dispersion at fixed size grows with size** for long lives; with short lives it is U-shaped, estimation noise
  dominating among small firms and floors among large ones.
- **The life length explains a growing share of it**, more than the degree, as for the floor itself.
- **Trimming fixed years steepens the decline and moves the ratios towards $q$**, monotonically for $n\ge20$, towards
  the plateau-only panel above, which data cannot locate.
- **Short lives push the ratios above $q$**; the bend appears from $n_{\min}\simeq5$.

## Summary

The floor of a firm's growth variance is the plateau fluctuation of a logistic unit in its field plus the climb from the
newborn size and the final collapse, divided by the life length. Lives end in dips of the field, at the Rice rate thinned by
the Feller extinction probability, so they are memoryless with a mean that grows steeply with the field's mean over its
amplitude. At fixed size the floor therefore varies mainly with the life length, and the ratio curve bends; the degree makes
this visible by keeping the floor of the larger, better connected firms large against their noise. All of it follows from a
single site in a Gaussian field. The two-time mean-field theory supplies its static statistics and the turnover;
its dynamic amplitude, which sets the level of the floor for large firms, still needs a sampler with windows longer than
the lifetimes.